#### Simple Gen AI APP Using Langchain

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
## Langsmith Tracking
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [2]:
# Clean text
def clean_text(text):
    lines = text.split("\n")  # Split into lines
    cleaned_lines = [line.strip() for line in lines if line.strip()]  # Remove empty lines and spaces
    cleaned_text = "\n".join(cleaned_lines)  # Join back
    return cleaned_text

In [3]:
## Data Ingestion--From the website we need to scrape the data
from langchain_community.document_loaders import TextLoader

loader=TextLoader('radius_QandA.txt')
docs=loader.load()
docs

[Document(metadata={'source': 'radius_QandA.txt'}, page_content='Frequently asked questions\nCommonly asked questions about best practices\nGeneral\nIs Kubernetes required to use Radius?\nCurrently yes. Although Radius is architected to run on any platform, today Kubernetes is the only hosting platform for Radius for the Radius control-plane and for containerized workloads. In the future, we plan to support other hosting platforms for serverless platforms.\n\nCan I incrementally adopt, or “try out” Radius?\nYes. The easiest way to add Radius to an existing application is through Radius annotations. Simply add the annotations to your existing Helm chart or Kubernetes YAML and you can use the Radius app graph, connections, and Recipes. Try the tutorial to learn more.\n\nDo I have to self-host Radius? Is there a managed service for Radius?\nOpen-source Radius requires that you self-host and run your own Radius instance in your Kubernetes cluster. In the future, we hope for providers to in

In [ ]:
# Apply cleanup to each document
for doc in docs:
    doc.page_content = clean_text(doc.page_content)

docs

[Document(metadata={'source': 'radius_QandA.txt'}, page_content='Frequently asked questions\nCommonly asked questions about best practices\nGeneral\nIs Kubernetes required to use Radius?\nCurrently yes. Although Radius is architected to run on any platform, today Kubernetes is the only hosting platform for Radius for the Radius control-plane and for containerized workloads. In the future, we plan to support other hosting platforms for serverless platforms.\nCan I incrementally adopt, or “try out” Radius?\nYes. The easiest way to add Radius to an existing application is through Radius annotations. Simply add the annotations to your existing Helm chart or Kubernetes YAML and you can use the Radius app graph, connections, and Recipes. Try the tutorial to learn more.\nDo I have to self-host Radius? Is there a managed service for Radius?\nOpen-source Radius requires that you self-host and run your own Radius instance in your Kubernetes cluster. In the future, we hope for providers to includ

In [5]:
### Load Data--> Docs-->Divide our Docuemnts into chunks dcouments-->text-->vectors-->Vector Embeddings--->Vector Store DB
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,       # smaller chunks for better retrieval relevance
    chunk_overlap=200,     # less repetition
    separators=["\n\n", "\n", ".", " "]  # explicit break preferences
)
documents=text_splitter.split_documents(docs)
documents

[Document(metadata={'source': 'radius_QandA.txt'}, page_content='Frequently asked questions\nCommonly asked questions about best practices\nGeneral\nIs Kubernetes required to use Radius?\nCurrently yes. Although Radius is architected to run on any platform, today Kubernetes is the only hosting platform for Radius for the Radius control-plane and for containerized workloads. In the future, we plan to support other hosting platforms for serverless platforms.\nCan I incrementally adopt, or “try out” Radius?\nYes. The easiest way to add Radius to an existing application is through Radius annotations. Simply add the annotations to your existing Helm chart or Kubernetes YAML and you can use the Radius app graph, connections, and Recipes. Try the tutorial to learn more.\nDo I have to self-host Radius? Is there a managed service for Radius?\nOpen-source Radius requires that you self-host and run your own Radius instance in your Kubernetes cluster. In the future, we hope for providers to includ

In [6]:
import re
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_text_splitters import TextSplitter


# Load file
with open("radius_QandA.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# Keep blank lines to identify Q&A breaks
lines = [line.rstrip() for line in raw_text.split("\n")]
cleaned_text = "\n".join(lines)

# Split into Q&A pairs — question is a line ending with "?"
qa_pairs = []
current_q = None
current_a = []
for line in cleaned_text.split("\n"):
    if line.strip().endswith("?"):
        if current_q:
            qa_pairs.append((current_q, "\n".join(current_a).strip()))
            current_a = []
        current_q = line.strip()
    elif current_q:
        current_a.append(line)

# Append last Q&A
if current_q:
    qa_pairs.append((current_q, "\n".join(current_a).strip()))

# Convert to Documents
qa_docs = [Document(page_content=f"Q: {q}\nA: {a}") for q, a in qa_pairs]

# Further split long answers if needed
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " "]
)
documents = splitter.split_documents(qa_docs)
documents

[Document(metadata={}, page_content='Q: Is Kubernetes required to use Radius?\nA: Currently yes. Although Radius is architected to run on any platform, today Kubernetes is the only hosting platform for Radius for the Radius control-plane and for containerized workloads. In the future, we plan to support other hosting platforms for serverless platforms.'),
 Document(metadata={}, page_content='Q: Can I incrementally adopt, or “try out” Radius?\nA: Yes. The easiest way to add Radius to an existing application is through Radius annotations. Simply add the annotations to your existing Helm chart or Kubernetes YAML and you can use the Radius app graph, connections, and Recipes. Try the tutorial to learn more.'),
 Document(metadata={}, page_content='Q: Do I have to self-host Radius? Is there a managed service for Radius?\nA: Open-source Radius requires that you self-host and run your own Radius instance in your Kubernetes cluster. In the future, we hope for providers to include Radius as a pa

In [7]:
from langchain_openai import OpenAIEmbeddings
embeddings=OpenAIEmbeddings()

In [8]:
from langchain_community.vectorstores import FAISS
vectorstoredb=FAISS.from_documents(documents,embeddings)
vectorstoredb

In [9]:
## Query From a vector db
query="How does Radius compare to Terraform"
result=vectorstoredb.similarity_search(query)
result[0].page_content

'Q: How does Radius compare to Terraform?\nA: Terraform is a tool for building, changing, and versioning infrastructure safely and efficiently. Terraform is a great tool for deploying infrastructure, but doesn’t provide a way to model an entire application and the dependencies between services and infrastructure, or act as an abstraction layer for multiple cloud providers.\n\nTeams looking to leverage existing Terraform modules can use Recipes to manage infrastructure provisioning, with the application defined in Bicep. The ability to define Radius applications in Terraform in addition to Bicep is on the roadmap.'

In [10]:
## Query From a vector db
query="How does compare to Waypoint"
result = vectorstoredb.similarity_search_with_score(query, k=2)

for doc, score in result:
    print(f"Score: {score}\nContent: {doc.page_content}\n")

Score: 0.3456381559371948
Content: Q: How does Radius compare to Waypoint?
A: HCP Waypoint is a HashiCorp-managed application deployment platform that simplifies the process of deploying applications into your infrastructure and helps you standardize your deployment process.

While Radius also is able to model and deploy applications, it also provides an application graph and Recipes to offer an end-to-end platform that is application-focused at every stage. Radius is more than an abstraction and deployment automation. The Radius app graph allows teams to understand their entire application and all the dependencies within it, even after an application is deployed. Recipes are also more than template automation. They provide abstraction + encapsulation so developers never are required to directly interact with cloud infrastructure templates or parameters.

Score: 0.47210514545440674
Content: Q: How does Radius compare to Backstage?
A: Backstage is an open platform for building developer

In [11]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-4o-mini",temperature=0)

In [12]:
## Retrieval Chain, Document chain

from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_template(
    """
Answer the following question based only on the provided context:
<context>
{context}
</context>


"""
)

document_chain=create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n\n\n'), additional_kwargs={})])
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x785038b3b0b0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x785038b38350>, root_client=<openai.OpenAI object at 0x78504b39e690>, root_async_client=<openai.AsyncOpenAI object at 0x785038b39760>, model_name='gpt-4o-mini', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'))
| StrOutputParser(), kwargs={},

However, we want the documents to first come from the retriever we just set up. That way, we can use the retriever to dynamically select the most relevant documents and pass those in for a given question.

In [13]:
### Input--->Retriever--->vectorstoredb

vectorstoredb

In [14]:
retriever=vectorstoredb.as_retriever()
from langchain.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,document_chain)
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x78503924ff80>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n\n\n'), additional_kwargs={})])
            | Chat

In [15]:
## Get the response form the LLM
response=retrieval_chain.invoke({"input":"How does Radius compare to .NET Aspire?"})
response['answer']

'Radius is an open-source project that allows for modeling, deploying, and managing applications across multiple cloud providers, while .NET Aspire is an opinionated, cloud-ready stack specifically for building .NET applications. Radius does not focus on local application runtime or processes, whereas .NET Aspire emphasizes the transition from local development to the cloud. Additionally, Radius provides tools for collaboration throughout the application lifecycle, such as the application graph and Recipes, which are not features of .NET Aspire.'

In [16]:

response

{'input': 'How does Radius compare to .NET Aspire?',
 'context': [Document(id='29ddfb20-a284-4e14-b6f7-554ba1d12e4e', metadata={}, page_content='Q: How does Radius compare to .NET Aspire?\nA: .NET Aspire is an opinionated, cloud ready stack for building .NET applications. .NET Aspire is delivered through a collection of NuGet packages that provide a batteries-included experience for building cloud-native applications as well as tools and IDE integration.\n\nWhere .NET Aspire is focused on the .NET experience from moving from local development with a debugger to the cloud, Radius is not opinionated about the application runtime and doesn’t seek to solve running applications locally as processes. Radius also offers tools for developers and operators to collaborate on an application throughout its lifecycle, such as the application graph and Recipes.'),
  Document(id='77e057d7-6ec4-40fa-a780-734d41014090', metadata={}, page_content='Q: How does Radius compare to Dapr?\nA: Dapr is a portab

In [17]:
response['context']

[Document(id='29ddfb20-a284-4e14-b6f7-554ba1d12e4e', metadata={}, page_content='Q: How does Radius compare to .NET Aspire?\nA: .NET Aspire is an opinionated, cloud ready stack for building .NET applications. .NET Aspire is delivered through a collection of NuGet packages that provide a batteries-included experience for building cloud-native applications as well as tools and IDE integration.\n\nWhere .NET Aspire is focused on the .NET experience from moving from local development with a debugger to the cloud, Radius is not opinionated about the application runtime and doesn’t seek to solve running applications locally as processes. Radius also offers tools for developers and operators to collaborate on an application throughout its lifecycle, such as the application graph and Recipes.'),
 Document(id='77e057d7-6ec4-40fa-a780-734d41014090', metadata={}, page_content='Q: How does Radius compare to Dapr?\nA: Dapr is a portable, event-driven runtime that makes it easy for developers to buil